# Lab 3

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Fourier Series

As you learned in lecture, there are many different (equivalent) representations of the fourier series approximation of a periodic function.

For no reason in particular, I'll use the following definition for an $N^{\textrm{th}}$ order fourier series:

$$f_N(t) = a_0 + \Sigma_{n = 1}^{N} \bigl[ a_n \cos(2 \pi n \omega_0 t) + b_n \sin(2 \pi n \omega_0 t) \bigr]$$

where:
$$P = \textrm{length of one period of the function}$$
$$a_0 = \frac{1}{P} \int_P f(t) dt$$ 
$$a_n = \frac{2}{P} \int_P f(t) \cos (2 \pi \frac{n}{P} t) dt$$
$$b_n = \frac{2}{P}\int_P f(t) \sin (2 \pi \frac{n}{P} t) dt$$

### Exercise 1 - Basic Fourier Series Implementation

#### Code

Implement `fourier_series(f, p, sr, N)` in the code box below. The arguments are as follows:

`f` - a numpy array of length `m` containing the values of $f(t)$

`p` - the length of one period **in units of time**

`sr` - number of array indices per second (sampling rate)

`N` - the order of the Nth order fourier series

Your function should return an array `fn` of length `m`, containing the $N^{\textrm{th}}$ order fourier series approximation of `f`.

In [12]:
def fourier_series(f, p, sr, N):
    ...

In [ ]:
# This box isn't graded, but you can use it to test your fourier series function
# Your approximation should get better as N increases!

sr = 100
len = 10
t = np.linspace(0, len, len * sr)

# Change this function to test your code!
f = 5 * (np.floor(t)%2)

for N in [1, 5, 50]:
    plt.plot(t, fourier_series(f, 2, sr, N), label=f"N = {N}")

plt.plot(t, f, label="f(t)")

plt.legend();

### Exercise 2 - Fourier Transform Intuition

#### Code

How do we evaluate the performance of the fourier series function? One way is to measure the squared error of the fourier approximation to the reference signal. If $f$ is the original function and $F$ is the fourier series approximation, then the error $E$ is:

$$E = \Sigma_t (F(t) - f(t))^2$$

Define a function `fourier_error` which computes the error $E$ with the following arguments:

`f` - a numpy array of length `m` containing the values of $f(t)$

`p_arr` - an array of period values of length `n`

`sr` - number of array indices per second (sampling rate)

`N` - the order of the Nth order fourier series

Your function should return `error`, an array of length `n`, which contains the net squared error between the fourier approximation and the original signal for each value of `p`, in order. You should call your `fourier_series` function within `fourier_error`.

**Note: due to numerical instability as error approaches 0, we caution modifying the test code below**

#### Written

Note that sinc is an aperiodic function. In a few sentences, explain how the fourier series could potentially be adapted to aperiodic functions. Justify your answer with a plot. 

In [18]:
def fourier_error(f, sr, p_arr, N=50):
    error = ...
    return error

Write your answer here.

In [ ]:
# This box isn't graded, but you can use it to test your fourier error function

sr = 100
len = 5
t = np.linspace(0, len, len * sr)

# Change this function to test your code!
f = np.sinc(t/10)
p_arr = 1/np.arange(1, 20)

plt.title("Error as 1/p increases")
plt.xlabel("Net Squared Error")
plt.ylabel("1/p")

plt.plot(fourier_error(f, sr, p_arr));

### Exercise 3 - A Digital Fourier Transform

#### Written

The Fourier Transform $F(\omega)$ of a signal $f(t)$, is defined:

$$F(\omega) = \int_{-\infty}^{\infty} f(t) e^{-j 2 \pi \omega t} dt$$

However, there are a number of problems with this expression. 

**For each of the subparts, we expect at most 3 lines of LaTeX. Your answers must all be equations, not partial expressions.**

1. In digital systems (computers), we cannot represent infinitely long signals. Correct the formula for $F(\omega)$ assuming that for any $t<0$, the function $f(t) = 0$ (making this a causal signal). For now, we'll keep the upper limit of the integral as $\infty$.

2. In a digital system, we cannot compute a continuous time integral - we don't actually have access to every point in time $f(t)$. Instead, we can sample the signal at some rate $r$, as we have with `np.linspace` in previous weeks. Let $n$ be used to index into the $n^{\textrm{th}}$ sample of $f$, sampled at a rate of $r$ samples per second. First, express $t$ in terms of $n$ and $r$. Then, approximate your $F(\omega)$ equation with a Riemann Sum so it can be computed on a digital system. Your final equation should not have any $t$ terms.

3. Suppose $f$ has only $N$ total samples - after all, any signal that we can store must be finite length. Modify your expression for $F(\omega)$ accordingly.

4. Define $\Omega = \frac{\omega}{r}$, and derive an expression for $F(\Omega)$ (you'll gain some intuition for why we do this in Exercise 6). This is (almost) the Discrete Time Fourier Transform (DTFT).

5. If we wanted to do some form of spectral analysis with the DTFT, we would have to pass in different values of $\Omega$ into the function. However, we cannot represent every single value of $\Omega$ (for the same reason that we wouldn't be able to express $f(t)$ as a continuous signal in (2)). One solution is to represent $F[\Omega]$ as a vector, just like $f[t]$. Given that $\Omega \in [0, N)$, derive an expression for $F[\Omega]$. **Keep in mind that $\Omega$ now represents an index, instead of a normalized frequency as it did in (4).**

6. You should still have a $\frac{1}{r}$ term in your $F[\Omega]$ expression. This scales all elements of the output by a constant, so it won't affect the relative "strengths" between frequencies. By removing this term, your $F[\Omega]$ expression should no longer have any dependence on the sampling rate. This is the Discrete Fourier Transform (DFT). Double check that your solution is equivalent to the one given in Exercise 5. 

7. Answer the following questions, given that you have a signal $f$ sampled over $N$ points at rate $r$.

    a. What is the maximum frequency ($\omega$, not $\Omega$) represented in your $F[\Omega]$ DFT? Why is this the case? 

    b. Suppose I want to improve the resolution of my DFT. Which quantity would I change, and how would I change it?

Write your response here: 

### Exercise 4 - Implementing the Discrete Fourier Transform

After your investigation in the previous section, you should have found the formula for the Discrete Fourier Transform:

$$F[\Omega] = \Sigma_{n = 0}^{N - 1} f[n] e^{- j 2 \pi n \frac{\Omega}{N}}$$

#### Code

Implement a function `dft(f)` with input `f`, which is a numpy array of length `N`. Your function should return a vector `ft` of length `N`, such that `ft[n] = F[n]`, using the formula given above. 

In [2]:
def dft(f):
    ...

### Exercise 5 - Fast Fourier Transform

Numpy as a very fast built-in function to compute the Discrete Fourier Transform called `np.fft.fft` - it has a very similar to your `dft` function that you defined above. We've given you some example code below to play with, but feel free to change it (we won't be grading it).

#### Written

1. Create a sine wave with some $\omega$ less than the sampling rate. Where would you expect the largest peak in the Continuous Time Fourier Transform? At what index does the largest peak appear in the DFT/FFT? Your answer should comment on the effect of changing both the length and sampling rate. 

2. Gradually increase $\omega$. What happens if $\omega$ is equal to half of your sampling rate? What happens if $\omega$ is greater than your sampling rate?

3. We won't discuss *why* the phenomenon you discovered occurs this week. However, we can (mostly) ignore this phenomenon by using `np.fft.rfft` instead of `np.fft.fft`. Explain what `np.fft.rfft` does internally.

Write your written response here.

In [ ]:
# This box isn't graded, but you can use it to play around with the fft/rfft 

sr = 20
t = 3
w = 5

t = np.linspace(0, 2 * np.pi, sr * t)
f = np.sin(w * t)

mag_fft = np.abs(np.fft.fft(f))

plt.stem(mag_fft);
plt.stem(np.abs(dft(f)));

### Exercise 6 - Inverse Transforms and Real Digital Filtering

In class, you learned that a Continuous Time Fourier Transform (CTFT) has an inverse transform which for brevity we can refer to has the iCTFT. If $F$ is the CTFT of a signal $f(t)$, then computing the iCTFT of $F$ should result in the original $f(t)$.

Similarly, the Fast Fourier Transform has an inverse transform called the Inverse Fast Fourier Transform (iFFT). Numpy gives us access to the following transform pairs: `np.fft.fft`/`np.fft.ifft`, and `np.fft.rfft`/`np.fft.irfft`. There are some limitations to these transforms which we will discuss near the end of the semester, if time permits. 

#### Code

1. Create `major_audio` from Lab 2. Your audio should be sampled at `sr = 20000` and should have a duration of three seconds.

2. Take the `rfft` of `major_audio`. Using the `rfft`, remove the "fifth" of the chord (that is, the pitch with frequency `440 * 3/2`).

3. Take the inverse transform of the `rfft`, and save this as `filtered_audio`. Verify that this sounds like the lower two notes of your chord.

#### Written

1. Evaluate your filter - you can do this visually by comparing your filtered audio to the superposition of sine waves (that is, constructing a chord with only the bottom two notes). What do you notice? What are some differences and similarities? You may find it easier to answer this question by plotting the DFT using the code from Exercise 6.

In [ ]:
# Write your code in here
from IPython.display import Audio
sr = 20000
time = 5
pitch = 440

major_audio = ...

filtered_audio = ...

Audio(major_audio, rate=sr)

In [ ]:
Audio(filtered_audio, rate=sr)

Write your written response here. 